# Environment and Hardware Verification

Inspects Python, PyTorch, CUDA GPUs, CPU architecture, system RAM, disk space, and ONNX Runtime providers across GPU training and CPU-only evaluation runtimes.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

REPO_ROOT = next((p for p in (Path('/kaggle/working/EdgeVisionInspector'), Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').exists()), Path('/kaggle/working/EdgeVisionInspector'))
os.environ['EDGEVISION_ROOT'] = str(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from edgevision.config import collect_runtime_environment
import json
import torch

env_info = collect_runtime_environment()
disk = shutil.disk_usage(str(REPO_ROOT))
env_info['disk_free_gb'] = round(disk.free / (1024 ** 3), 3)
print(json.dumps(env_info, indent=2))

In [ ]:
# Verify each visible CUDA device with a small tensor allocation and forward pass.
if torch.cuda.is_available():
    print('Visible CUDA devices:', torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        dev = torch.device(f'cuda:{idx}')
        x = torch.randn((1, 3, 320, 320), device=dev)
        conv = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1).to(dev)
        y = conv(x)
        torch.cuda.synchronize(dev)
        print(f'Device {idx} ({torch.cuda.get_device_name(idx)}): output={tuple(y.shape)} OK')
else:
    print('Running in CPU-only mode.')